## CS310 Natural Language Processing
## Lab 11: In-Context Learning and Prompting

In this lab, we will practice some in-context learning techniques, such as few-shot learning and chain-of-thought prompting, for solving QA problems.

## T1. Run LLMs locally

### Step 1) Install llama.cpp

Build the [llama.cpp](https://github.com/ggml-org/llama.cpp) tool, or download the binaries from the [release page](https://github.com/ggml-org/llama.cpp/releases).

---

### Step 1) Download model

We are going to download a quantized [Qwen3-1.7B model](https://huggingface.co/Qwen/Qwen3-1.7B-GGUF), whose format is converted to `gguf`.

- Using the `huggingface-cli` or [`hfd`](https://hf-mirror.com/) tools.

A quick command to download the model is:

```bash
huggingface-cli download Qwen/Qwen3-1.7B-GGUF
```


In case you want to compare the performance with an older model, use the following command to download Qwen2.5-7B-Chat-GGUF:

- Following the tutorial here: [Qwen2.5-7B-Instruct-GGUF](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct-GGUF)

```bash
huggingface-cli download Qwen/Qwen2.5-7B-Instruct-GGUF --include "qwen2.5-7b-instruct-q5_k_m*.gguf" --local-dir . --local-dir-use-symlinks False
```

---


### Step 3) Run model

After you have built `llama.cpp`, you can run the model with following command:

```bash
.\llama-b9114-bin-win-cpu-x64\llama-cli.exe -m .\models\Qwen3-1.7B-GGUF\<这里替换为实际的.gguf文件名>
```

Then you can start interacting with the model in command line. Try to solve the following problems.
 - Use zero-shot and few-shot prompting to solve the problems.
 - Add Chain-of-Thought prompt if needed.
 - For Qwen3-1.7B, append `\nothink` to disable reasoning model.


Try solving these problems with prompting:
1. Q: A juggler can juggle 16 balls. Half of the balls are golf balls, and half of the golf balls are blue. How many blue golf balls are there? A: 
2. 鸡和兔在一个笼子里，共有35个头，94只脚，那么鸡有多少只，兔有多少只？
3. Q: 242342 + 423443 = ? A: 
4. 一个人花8块钱买了一只鸡，9块钱卖掉了，然后他觉得不划算，花10块钱又买回来了，11块卖给另外一个人。问他赚了多少?

---

## T2. Practice few-shot prompting

For the second pratice, we will run some few-shot prompting experiments in Python.

You need to first download the original [Qwen2.5-7B](https://huggingface.co/Qwen/Qwen2.5-7B) and [Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B) model weights.

Also, the dataset used is [MMLU](https://huggingface.co/datasets/cais/mmlu). You need to download the zip file and extract it to the `./MMLU` folder.

In [1]:
from transformers import AutoTokenizer,AutoModelForCausalLM
import torch
import json
import numpy as np
import pprint

e:\CS310-Natural-Language-Processing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


First, define some helper functions for constructing prompts and running inference.

In [2]:
choices = ["A", "B", "C", "D"]

def format_subject(subject):
    l = subject.split("_")
    s = ""
    for entry in l:
        s += " " + entry
    return s

def format_example(input_list):
    prompt = input_list[0]
    k = len(input_list) - 2
    for j in range(k):
        prompt += "\n{}. {}".format(choices[j], input_list[j+1])
    prompt += "\nAnswer:"
    return prompt

def format_shots(prompt_data):
    prompt = ""
    for data in prompt_data:
        prompt += data[0]
        k = len(data) - 2
        for j in range(k):
            prompt += "\n{}. {}".format(choices[j], data[j+1])
        prompt += "\nAnswer:"
        prompt += data[k+1] + "\n\n"

    return prompt

def gen_prompt(input_list, subject, prompt_data):
    prompt = "The following are multiple choice questions (with answers) about {}.\n\n".format(
        format_subject(subject)
    )
    prompt += format_shots(prompt_data)
    prompt += format_example(input_list)
    return prompt

The following `inference()` function constructs the full input by prepending the few-shot examples to the `input_text`, and generate **one** token as the output, because the task modality is multiple choice question.

Hint:
- Use `model.generate()` to generate the next token.
- Specify `max_new_tokens=1`, `do_sample=False`, `output_scores=True`, and `return_dict_in_generate=True`.
- To compute the probabilities of all answer choices (`["A", "B", "C", "D"]`), first collect the logits from the output, and then apply softmax on them. 

In [3]:
def inference(model, tokenizer, input_text, subject, prompt_data):
    """
    Run inference on MMLU input with few-shot examples.
    """
    if len(prompt_data) > 0:
        full_input = gen_prompt(input_text, subject, prompt_data) # add few-shot examples
    else:
        full_input = input_text

    # 1. Run tokenizer on full_input
    ### START YOUR CODE ###
    inputs = tokenizer(full_input, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    ### END YOUR CODE ###

    # 2. Generate one next token and get the logits
    ### START YOUR CODE ###
    outputs = model.generate(
        **inputs,
        max_new_tokens=1,
        do_sample=False,
        output_scores=True,
        return_dict_in_generate=True
    )
    logits = outputs.scores[0][0]
    ### END YOUR CODE ###

    # 3. Compute the probability of each choice ["A", "B", "C", "D"]
    ### START YOUR CODE ###
    # Hint: probs should be a 4-element numpy array.
    choice_logits = []
    for c in choices:
        token_ids = tokenizer(c, add_special_tokens=False).input_ids
        token_ids_with_space = tokenizer(" " + c, add_special_tokens=False).input_ids

        candidate_logits = []
        if len(token_ids) == 1:
            candidate_logits.append(logits[token_ids[0]])
        if len(token_ids_with_space) == 1:
            candidate_logits.append(logits[token_ids_with_space[0]])

        if len(candidate_logits) == 0:
            candidate_logits.append(logits[token_ids[0]])

        choice_logits.append(torch.max(torch.stack(candidate_logits)))

    probs = torch.softmax(torch.stack(choice_logits), dim=0).detach().cpu().numpy()
    ### END YOUR CODE ###

    output_text = {0: "A", 1: "B", 2: "C", 3: "D"}[np.argmax(probs)]
    confidence = np.max(probs)

    return output_text, full_input, confidence.item()

In [5]:
qwen2_model_path = '../lab8/gpt2-mini'

qwen2_tokenizer = AutoTokenizer.from_pretrained(qwen2_model_path,
                                          use_fast=True,
                                          unk_token="<unk>",
                                          bos_token="<s>", eos_token="</s>",
                                          add_bos_token=False)

qwen2_model = AutoModelForCausalLM.from_pretrained(qwen2_model_path)


Loading weights: 100%|██████████| 52/52 [00:00<00:00, 11845.74it/s]


In [6]:
qwen3_model_path = '../lab8/Qwen3-0.6B-Base'
qwen3_tokenizer = AutoTokenizer.from_pretrained(qwen3_model_path)
qwen3_model = AutoModelForCausalLM.from_pretrained(qwen3_model_path)


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 2547.25it/s]


---

Next, load the MMLU dataset. For simplicity, we only use a small fraction, i.e., the in-domain (ID) test set. 

In [7]:
data = {}
prompt = {} # The *.json include few-shot examples.

with open(f"./MMLU/MMLU_ID_test.json",'r') as f:
    data = json.load(f)
    
with open(f"./MMLU/MMLU_ID_prompt.json",'r') as f:
    prompt = json.load(f)

The data is organized by subjects.

In [8]:
print(data.keys())

print()
pprint.pp(data['high_school_mathematics'][3])

dict_keys(['abstract_algebra', 'anatomy', 'astronomy', 'business_ethics', 'clinical_knowledge', 'college_biology', 'college_chemistry', 'college_computer_science', 'college_mathematics', 'college_medicine', 'college_physics', 'computer_security', 'conceptual_physics', 'econometrics', 'electrical_engineering', 'elementary_mathematics', 'formal_logic', 'global_facts', 'high_school_biology', 'high_school_chemistry', 'high_school_computer_science', 'high_school_european_history', 'high_school_geography', 'high_school_government_and_politics', 'high_school_macroeconomics', 'high_school_mathematics', 'high_school_microeconomics', 'high_school_physics'])

['At breakfast, lunch, and dinner, Joe randomly chooses with equal '
 'probabilities either an apple, an orange, or a banana to eat. On a given '
 'day, what is the probability that Joe will eat at least two different kinds '
 'of fruit?',
 '\\frac{7}{9}',
 '\\frac{8}{9}',
 '\\frac{5}{9}',
 '\\frac{9}{11}',
 'B']


Few-shot prompts also come in subjects, and each subject has a list of 5 examples.

In [9]:
print(len(prompt['high_school_mathematics']))
print(len(prompt['high_school_physics']))

5
5


We stick to one subject, `high_school_mathematics` for this example.

In [10]:
subject = 'high_school_mathematics'
data_sub = data[subject]
prompt_sub = prompt[subject]

Take one input example and generate the full prompt by calling `gen_prompt()`

In [11]:
input_text = data_sub[3]
prompt_text = gen_prompt(input_text, subject, prompt_sub)
print(prompt_text)

The following are multiple choice questions (with answers) about  high school mathematics.

Joe was in charge of lights for a dance. The red light blinks every two seconds, the yellow light every three seconds, and the blue light every five seconds. If we include the very beginning and very end of the dance, how many times during a seven minute dance will all the lights come on at the same time? (Assume that all three lights blink simultaneously at the very beginning of the dance.)
A. 3
B. 15
C. 6
D. 5
Answer:B

Five thousand dollars compounded annually at an $x\%$ interest rate takes six years to double. At the same interest rate, how many years will it take $\$300$ to grow to $\$9600$?
A. 12
B. 1
C. 30
D. 5
Answer:C

The variable $x$ varies directly as the square of $y$, and $y$ varies directly as the cube of $z$. If $x$ equals $-16$ when $z$ equals 2, what is the value of $x$ when $z$ equals $\frac{1}{2}$?
A. -1
B. 16
C. -\frac{1}{256}
D. \frac{1}{16}
Answer:C

Simplify and write th

OK, now let's run the few-shot inference. It will take a while.

In [14]:
output, _, conf = inference(qwen3_model, qwen3_tokenizer, input_text, subject, prompt_sub)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


In [15]:
# Test
print(output)
print(conf)

# You are expected to see the following output:
# B

B
0.3447858989238739


In [16]:
output, _, conf = inference(qwen3_model, qwen3_tokenizer, input_text, subject, prompt_sub)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


In [17]:
# Test
print(output)
print(conf)

# You are expected to see the following output:
# B

B
0.3447858989238739


Let's test with zero-shot prompting to see if the answers are correct.

In [18]:
zs_prompt = '''
    At breakfast, lunch, and dinner, Joe randomly chooses with equal probabilities either an apple, an orange, or a banana to eat. On a given day, what is the probability that Joe will eat at least two different kinds of fruit?
    A. \frac{7}{9}
    B. \frac{8}{9}
    C. \frac{5}{9}
    D. \frac{9}{11}
    Answer:
'''

In [19]:
output, _, conf = inference(qwen2_model, qwen2_tokenizer, zs_prompt, subject, prompt_data=[])
print(output)
print(conf)

A
0.5707968473434448


In [20]:
output, _, conf = inference(qwen3_model, qwen3_tokenizer, zs_prompt, subject, prompt_data=[])
print(output)
print(conf)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


A
0.8664913177490234
